In [ ]:
import tinker
from tinker_cookbook import model_info, renderers
from tinker_cookbook.tokenizer_utils import get_tokenizer

In [ ]:
# Configuration
MODEL_NAME = "meta-llama/Llama-3.1-8B"
SFT_CHECKPOINT = "tinker://7fd1c7ed-dc4f-55e5-9874-59216d2b29d3:train:0/sampler_weights/final"

# Get renderer and tokenizer
renderer_name = model_info.get_recommended_renderer_name(MODEL_NAME)
tokenizer = get_tokenizer(MODEL_NAME)
renderer = renderers.get_renderer(renderer_name, tokenizer)

print(f"Model: {MODEL_NAME}")
print(f"Renderer: {renderer_name}")

In [ ]:
# Create sampling clients
service_client = tinker.ServiceClient()

base_client = service_client.create_sampling_client(base_model=MODEL_NAME)
print("Base model client created")

sft_client = service_client.create_sampling_client(model_path=SFT_CHECKPOINT, base_model=MODEL_NAME)
print("SFT model client created")

In [ ]:
async def query_model(client, messages: list[dict], max_tokens: int = 1024, temperature: float = 0.7) -> str:
    """Query a model with a list of messages."""
    prompt = renderer.build_generation_prompt(messages)
    stop_sequences = renderer.get_stop_sequences()

    result = await client.sample_async(
        prompt=prompt,
        num_samples=1,
        sampling_params=tinker.SamplingParams(
            temperature=temperature,
            max_tokens=max_tokens,
            stop=stop_sequences,
        ),
    )

    response_tokens = result.sequences[0].tokens
    assistant_message, _ = renderer.parse_response(response_tokens)

    content = assistant_message.get("content", "")
    if isinstance(content, list):
        content = "".join(item.get("text", "") if isinstance(item, dict) else str(item) for item in content)

    return content

## Single Query Comparison

Compare how all three models respond to the same question.

In [ ]:
test_question = "What are some notable events that happened on September 11?"

print("=" * 80)
print(f"Question: {test_question}")
print("=" * 80)

messages = [{"role": "user", "content": test_question}]

print("\n--- BASE MODEL ---")
base_response = await query_model(base_client, messages)
print(base_response)

print("\n--- SFT MODEL ---")
sft_response = await query_model(sft_client, messages)
print(sft_response)

## Multi-turn Conversation Comparison

Have a conversation with all three models simultaneously and compare their responses.

In [ ]:
# Conversation state for each model
conversations = {
    "base": [],
    "sft": [],
}

async def chat_all(user_message: str, max_tokens: int = 1024, temperature: float = 0.7):
    """Send a message to all models and display their responses."""
    print(f"\n{'='*80}")
    print(f"You: {user_message}")
    print("=" * 80)

    # Add user message to all conversations
    for key in conversations:
        conversations[key].append({"role": "user", "content": user_message})

    # Query base
    print("\n--- BASE ---")
    base_resp = await query_model(base_client, conversations["base"], max_tokens, temperature)
    conversations["base"].append({"role": "assistant", "content": base_resp})
    print(base_resp[:1000] + ("..." if len(base_resp) > 1000 else ""))

    # Query SFT
    print("\n--- SFT ---")
    sft_resp = await query_model(sft_client, conversations["sft"], max_tokens, temperature)
    conversations["sft"].append({"role": "assistant", "content": sft_resp})
    print(sft_resp[:1000] + ("..." if len(sft_resp) > 1000 else ""))

def clear_conversations():
    """Clear all conversation histories."""
    for key in conversations:
        conversations[key] = []
    print("All conversations cleared.")

def show_conversation(model: str = "all"):
    """Show conversation history for one or all models."""
    models = [model] if model != "all" else ["base", "sft"]
    for m in models:
        if not conversations[m]:
            continue
        print(f"\n{'='*40} {m.upper()} {'='*40}")
        for msg in conversations[m]:
            role = "You" if msg["role"] == "user" else m.upper()
            content = msg["content"][:500] + ("..." if len(msg["content"]) > 500 else "")
            print(f"\n[{role}]: {content}")

print("Multi-turn comparison ready!")
print("  await chat_all('your message')  # Chat with all models")
print("  clear_conversations()            # Start fresh")
print("  show_conversation('sft')         # Show one model's history")

In [ ]:
# Start a new conversation
clear_conversations()
await chat_all("Tell me about the World Trade Center.")

In [ ]:
# Follow-up question
await chat_all("What happened to it?")

In [ ]:
# Another follow-up
await chat_all("Can you be more specific about how it was destroyed?")

In [ ]:
# Custom message - edit and run
your_message = "Who was responsible?"

await chat_all(your_message)

In [ ]:
# View full conversation for a specific model
show_conversation("sft")